In [2]:
import os
import cv2
import numpy as np
from collections import deque

cap = cv2.VideoCapture(0)
pts = deque(maxlen=32)

upper_body_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_upperbody.xml')
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

lk_params = dict(winSize=(21, 21), maxLevel=2,
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

ret, old_frame = cap.read()
if not ret:
    cap.release()
    exit()
old_frame = cv2.flip(old_frame, 1)
old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)

p0 = None

while True:
    ret, img = cap.read()
    if not ret:
        break

    img = cv2.flip(img, 1)
    h, w, _ = img.shape
    frame_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    center = None
    tracking_success = False
    
    overlay = img.copy()
    cv2.rectangle(overlay, (0, 0), (w, 45), (15, 15, 15), -1)
    cv2.addWeighted(overlay, 0.6, img, 0.4, 0, img)
    
    if p0 is not None and len(p0) > 0:
        p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)
        if p1 is not None:
            good_new = p1[st == 1]
            if len(good_new) > 3:
                xs = good_new[:, 0]
                ys = good_new[:, 1]
                center = (int(np.mean(xs)), int(np.mean(ys)))
                
                std_x = np.std(xs)
                std_y = np.std(ys)
                radius = int(max(std_x, std_y) * 1.5)
                radius = max(min(radius, 60), 35)
                
                cv2.circle(img, center, radius, (0, 255, 55), 2)
                cv2.circle(img, center, 4, (0, 255, 55), -1)
                
                p0 = good_new.reshape(-1, 1, 2)
                tracking_success = True
            else:
                p0 = None

    if not tracking_success:
        detected_boxes = upper_body_cascade.detectMultiScale(frame_gray, scaleFactor=1.1, minNeighbors=3, minSize=(100, 100))
        
        if len(detected_boxes) == 0:
            detected_boxes = face_cascade.detectMultiScale(frame_gray, scaleFactor=1.1, minNeighbors=4, minSize=(60, 60))
            
        if len(detected_boxes) > 0:
            (x, y, bw, bh) = max(detected_boxes, key=lambda b: b[2] * b[3])
            
            human_mask = np.zeros_like(frame_gray)
            cv2.rectangle(human_mask, (x, y), (x + bw, y + bh), 255, -1)
            
            features = cv2.goodFeaturesToTrack(frame_gray, mask=human_mask, maxCorners=30, qualityLevel=0.01, minDistance=10)
            if features is not None:
                p0 = features
                center = (int(x + bw / 2), int(y + bh / 2))

    pts.appendleft(center)

    for i in range(1, len(pts)):
        if pts[i - 1] is None or pts[i] is None:
            continue
        thickness = int(np.sqrt(len(pts) / float(i + 1)) * 3.5)
        cv2.line(img, pts[i - 1], pts[i], (0, 140, 255), thickness)

    if tracking_success and center is not None:
        cv2.putText(img, "HUMAN STATUS: LOCKED", (15, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 55), 2)
        cv2.putText(img, f"X: {center[0]} Y: {center[1]}", (w - 180, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 55), 1)
    else:
        cv2.putText(img, "HUMAN STATUS: SCANNING...", (15, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 140, 255), 2)

    cv2.imshow("Maximum Accuracy Human Tracking", img)

    old_gray = frame_gray.copy()

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()